# Real-World EDA Project
Complete analysis: question -> insight -> recommendation.

In [ ]:
import micropip
await micropip.install(['pandas','matplotlib','numpy','seaborn'])
print('Ready!')

## Project: Student Performance Analysis

**Business Question:** What factors predict student academic success and what should the institution prioritise?

**Approach:** Load data -> EDA -> Visualise -> Insights -> Recommendations

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'gender':         np.random.choice(['M','F'], n, p=[0.48,0.52]),
    'parental_edu':   np.random.choice(['school','college','bachelor','master','phd'], n, p=[0.2,0.25,0.3,0.15,0.1]),
    'study_hours':    np.random.gamma(2, 3, n).clip(0, 20).round(1),
    'internet':       np.random.choice(['yes','no'], n, p=[0.85,0.15]),
    'absences':       np.random.poisson(5, n),
    'prev_grade':     np.random.normal(13, 3, n).clip(0, 20).round(1),
    'health':         np.random.randint(1, 6, n),
})
df['final_score'] = (
    0.3 * df['study_hours'] + 0.5 * df['prev_grade'] +
    np.where(df['internet']=='yes', 0.5, -0.5) - 0.15*df['absences'] +
    np.random.randn(n)*3
).clip(0, 20).round(1)
df['passed'] = (df['final_score'] >= 10).astype(int)
print(f"Dataset: {df.shape}")
print(f"Pass rate: {df['passed'].mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].hist(df['final_score'], bins=25, color='#6b21a8', alpha=0.8, edgecolor='white')
axes[0].axvline(10, color='#dc2626', linestyle='--', lw=2, label='Pass threshold')
axes[0].set_title('Final Score Distribution'); axes[0].legend()
axes[1].bar(['Fail','Pass'], [df['passed'].value_counts()[0], df['passed'].value_counts()[1]],
            color=['#dc2626','#059669'], alpha=0.8)
axes[1].set_title('Pass/Fail Balance')
corr = df[['study_hours','absences','prev_grade','health','final_score']].corr()
im = axes[2].imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
cols = list(corr.columns)
axes[2].set_xticks(range(len(cols))); axes[2].set_xticklabels(cols, rotation=35, ha='right', fontsize=8)
axes[2].set_yticks(range(len(cols))); axes[2].set_yticklabels(cols, fontsize=8)
for i in range(len(corr)):
    for j in range(len(cols)):
        axes[2].text(j,i,f"{corr.iloc[i,j]:.2f}",ha='center',va='center',fontsize=7)
axes[2].set_title('Correlations')
plt.suptitle('Overview', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0,0].scatter(df['study_hours'], df['final_score'],
    c=df['passed'], cmap='RdYlGn', alpha=0.4, s=15)
axes[0,0].set_xlabel('Study Hours'); axes[0,0].set_ylabel('Final Score')
axes[0,0].set_title('Study Hours vs Score')
axes[0,1].scatter(df['absences'], df['final_score'],
    c=df['passed'], cmap='RdYlGn', alpha=0.4, s=15)
axes[0,1].set_xlabel('Absences'); axes[0,1].set_ylabel('Final Score')
axes[0,1].set_title('Absences vs Score')
edu_order = ['school','college','bachelor','master','phd']
edu_pass = df.groupby('parental_edu')['passed'].mean().reindex(edu_order)
axes[1,0].bar(edu_order, edu_pass, color='#6b21a8', alpha=0.8)
axes[1,0].set_title('Pass Rate by Parental Education')
axes[1,0].tick_params(axis='x', rotation=20)
internet_pass = df.groupby('internet')['passed'].mean()
axes[1,1].bar(internet_pass.index, internet_pass, color=['#dc2626','#059669'], alpha=0.8)
axes[1,1].set_xlabel('Internet Access'); axes[1,1].set_ylabel('Pass Rate')
axes[1,1].set_title('Pass Rate by Internet Access')
plt.suptitle('What Drives Student Success?', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
print("KEY INSIGHTS:")
print(f"  Study <3hrs:  {df[df['study_hours']<3]['passed'].mean()*100:.0f}% pass vs {df[df['study_hours']>=3]['passed'].mean()*100:.0f}% for 3+hrs")
print(f"  >10 absences: {df[df['absences']>10]['passed'].mean()*100:.0f}% pass rate")
print(f"  Internet gap: {(df[df['internet']=='yes']['passed'].mean()-df[df['internet']=='no']['passed'].mean())*100:.1f}% higher pass rate")
print("\nRECOMMENDATIONS:")
print("  1. Study skills programme for students studying <3hrs/week")
print("  2. Alert system for students with >5 absences")
print("  3. Device/internet loans for at-risk students")
print("  4. Early intervention for students with prev grade < 12")

---
**Your turn:** Analyse `health` score vs `final_score`. Does student health significantly impact performance?